# 문진톡톡 Q1 파이프라인 테스트 — gold leakage 분리 버전

이 노트북은 Q1 증상 문진 파이프라인을 **세 가지 모드**로 분리해 검증합니다.

```text
[Stage 1] 환자 발화 원문 보존
[Stage 2] phrase span 추출
[Stage 2.5] 사투리·구어체 → 표준어 변환 + 표준 증상 후보 생성
[Stage 3] 전체 symptom vocabulary와 Ensemble Retrieval
[Stage 4] LLM Verifier 또는 local verifier 검증
[Stage 5] safety_flag 병렬 감지
[Stage 6] 원페이퍼 입력 JSON 생성
```

## 핵심 수정점

이전 노트북은 로컬 모드에서 테스트셋의 `spans`, `normalized_candidates`를 inference 입력으로 사용할 수 있었습니다. 그것은 **retrieval component oracle test**로는 유용하지만, end-to-end 성능으로 해석하면 안 됩니다.

이번 버전은 아래처럼 명확히 분리합니다.

| 모드 | inference 때 쓰는 값 | 목적 |
|---|---|---|
| `LOCAL_INPUT_ONLY` | `input`만 사용 | gold leakage 없는 로컬 smoke test |
| `ORACLE_RETRIEVAL_COMPONENT` | gold spans/normalization 사용 | Stage 3 retrieval 함수만 점검 |
| `AWS_E2E` | `input`만 사용 | Bedrock/Titan 실제 end-to-end 테스트 |

기본값은 `LOCAL_INPUT_ONLY`입니다. 실제 평가용은 `AWS_E2E`입니다.

In [ ]:
import boto3
import json
import os

AWS_PROFILE = "munjin"
AWS_REGION = "ap-northeast-2"

session = boto3.Session(
    profile_name=AWS_PROFILE,
    region_name=AWS_REGION
)

sts = session.client("sts")
identity = sts.get_caller_identity()

print("AWS 연결 확인")
print("Account:", identity["Account"])
print("Arn:", identity["Arn"])

bedrock = session.client("bedrock")
bedrock_runtime = session.client("bedrock-runtime")

print("Bedrock clients ready")

In [ ]:
LLM_MODEL_ID = "apac.amazon.nova-lite-v1:0"

response = bedrock_runtime.converse(
    modelId=LLM_MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": "한 문장으로만 답해줘. 대한민국의 수도는?"
                }
            ]
        }
    ],
    inferenceConfig={
        "temperature": 0,
        "maxTokens": 128
    }
)

print(response["output"]["message"]["content"][0]["text"])

In [ ]:
EMBEDDING_MODEL_ID = "amazon.titan-embed-text-v2:0"

body = {
    "inputText": "기침이 나고 목에 가래가 낀 것 같아요",
    "dimensions": 512,
    "normalize": True
}

response = bedrock_runtime.invoke_model(
    modelId=EMBEDDING_MODEL_ID,
    body=json.dumps(body),
    accept="application/json",
    contentType="application/json"
)

result = json.loads(response["body"].read())
embedding = result["embedding"]

print("embedding length:", len(embedding))
print("first 5 values:", embedding[:5])

## 0. 설치

처음 한 번만 실행하면 됩니다. `boto3`는 AWS Bedrock 호출용입니다.

In [ ]:

import sys, subprocess, importlib.util

packages = [
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("boto3", "boto3"),
    ("tqdm", "tqdm"),
]

for pip_name, import_name in packages:
    if importlib.util.find_spec(import_name) is None:
        print(f"[INSTALL] {pip_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
    else:
        print(f"[OK] {pip_name}")

## 1. 설정

처음에는 `PIPELINE_MODE = "LOCAL_INPUT_ONLY"`로 실행합니다. 이 모드는 **답지 span/정규화 후보를 inference에 넣지 않습니다.**

AWS 테스트는 아래 순서로 진행하세요.

1. `PIPELINE_MODE = "AWS_E2E"`, `USE_AWS_STAGE2=True`, 나머지 False
2. `USE_AWS_EMBEDDINGS=True` 추가
3. `USE_AWS_VERIFIER=True` 추가
4. `MAX_CASES`를 3 → 10 → 30 → None 순서로 확대

In [ ]:

from pathlib import Path
import os, json, re, math, time, hashlib
from collections import Counter, defaultdict
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# -----------------------------
# 파일 경로
# -----------------------------
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
CACHE_DIR = Path("cache")
OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

TEST_CASES_PATH = DATA_DIR / "q1_gangwon_test_cases_100.json"
SYMPTOM_INDEX_PATH = DATA_DIR / "symptom_index.json"

# -----------------------------
# 실행 모드
# -----------------------------
# LOCAL_INPUT_ONLY: input만 사용. gold span/normalization 사용 금지.
# ORACLE_RETRIEVAL_COMPONENT: gold span/normalization 사용. Stage 3 retrieval component test 전용.
# AWS_E2E: input만 사용. Bedrock/Titan/Verifier를 단계별로 켜서 실제 테스트.
PIPELINE_MODE = "AWS_E2E"

MAX_CASES = None          # 처음에는 5~10개 추천. 전체는 None.
TOP_K = 5
AWS_SLEEP_SEC = 0.15

# -----------------------------
# AWS 연결 토글
# -----------------------------
USE_AWS_STAGE2 = True       # Stage 2/2.5 Bedrock Tool Use
USE_AWS_EMBEDDINGS = True   # Stage 3 Titan Text Embeddings V2
USE_AWS_VERIFIER = True     # Stage 4 Bedrock verifier

if PIPELINE_MODE == "AWS_E2E":
    # 필요에 따라 하나씩 True로 바꾸세요.
    USE_AWS_STAGE2 = True
    # USE_AWS_EMBEDDINGS = True
    # USE_AWS_VERIFIER = True

# -----------------------------
# AWS 설정
# -----------------------------
AWS_REGION = os.getenv("AWS_REGION", "ap-northeast-2")
LLM_MODEL_ID = os.getenv("BEDROCK_LLM_MODEL_ID", "apac.amazon.nova-lite-v1:0")
EMBEDDING_MODEL_ID = os.getenv("BEDROCK_EMBEDDING_MODEL_ID", "amazon.titan-embed-text-v2:0")
EMBEDDING_DIMENSIONS = int(os.getenv("BEDROCK_EMBEDDING_DIMENSIONS", "512"))

assert PIPELINE_MODE in {"LOCAL_INPUT_ONLY", "ORACLE_RETRIEVAL_COMPONENT", "AWS_E2E"}
print("PIPELINE_MODE:", PIPELINE_MODE)
print("USE_AWS_STAGE2:", USE_AWS_STAGE2)
print("USE_AWS_EMBEDDINGS:", USE_AWS_EMBEDDINGS)
print("USE_AWS_VERIFIER:", USE_AWS_VERIFIER)
print("AWS_REGION:", AWS_REGION)

## 2. 데이터 로드 및 `symptom_vocab` 생성

`symptom_index.json`은 reverse index입니다.

```text
증상명 → 이 증상이 등장한 아산백과 질환 항목 목록
```

런타임 retrieval에는 질환 목록을 쓰지 않고, key 목록만 사용합니다.

```python
symptom_vocab = sorted(symptom_index.keys())
```

In [ ]:

def load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

raw_cases = load_json(TEST_CASES_PATH)
if isinstance(raw_cases, dict) and "cases" in raw_cases:
    test_cases = raw_cases["cases"]
    dataset_meta = raw_cases.get("metadata", {})
else:
    test_cases = raw_cases
    dataset_meta = {}

symptom_index = load_json(SYMPTOM_INDEX_PATH)
symptom_vocab = sorted(symptom_index.keys())
symptom_set = set(symptom_vocab)

print("테스트 케이스 수:", len(test_cases))
print("symptom_vocab 수:", len(symptom_vocab))
print("예시 symptom_vocab:", symptom_vocab[:20])

missing_expected = []
for case in test_cases:
    for field in ["expected_symptoms", "expected_denied", "expected_needs_review"]:
        for x in case.get(field, []):
            if x not in symptom_set:
                missing_expected.append((case.get("case_id"), field, x))

print("symptom_vocab에 없는 expected label 수:", len(missing_expected))
if missing_expected[:20]:
    display(pd.DataFrame(missing_expected[:20], columns=["case_id", "field", "missing_label"]))

preview_df = pd.DataFrame([{
    "case_id": c.get("case_id"),
    "input": c.get("input"),
    "expected_symptoms": ", ".join(c.get("expected_symptoms", [])),
    "expected_denied": ", ".join(c.get("expected_denied", [])),
    "expected_safety_flags": ", ".join(c.get("expected_safety_flags", [])),
} for c in test_cases[:10]])
display(preview_df)

## 3. 로컬 retrieval 유틸

Stage 3의 후보 검색 함수입니다. 이 함수는 질환 예측을 하지 않습니다. 오직 `symptom_vocab` 87개 key와 입력 query를 비교합니다.

In [ ]:

def normalize_text(x: Any) -> str:
    if x is None:
        return ""
    x = str(x).replace("​", " ").replace(" ", " ")
    x = re.sub(r"\s+", " ", x)
    return x.strip()


def char_ngrams(text: str, n: int = 2) -> List[str]:
    text = normalize_text(text)
    text = re.sub(r"\s+", "", text)
    if not text:
        return []
    if len(text) <= n:
        return [text]
    return [text[i:i+n] for i in range(len(text)-n+1)]


def token_units(text: str) -> List[str]:
    text = normalize_text(text)
    words = [w for w in re.split(r"\s+", text) if w]
    return words + char_ngrams(text, 2)


def jaccard(a: List[str], b: List[str]) -> float:
    sa, sb = set(a), set(b)
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def cosine_counter(a: Counter, b: Counter) -> float:
    if not a or not b:
        return 0.0
    keys = set(a) | set(b)
    dot = sum(a.get(k, 0) * b.get(k, 0) for k in keys)
    na = math.sqrt(sum(v*v for v in a.values()))
    nb = math.sqrt(sum(v*v for v in b.values()))
    if na == 0 or nb == 0:
        return 0.0
    return dot / (na * nb)


def exact_score(query: str, key: str) -> float:
    return 1.0 if normalize_text(query) == normalize_text(key) else 0.0


def substring_score(query: str, key: str) -> float:
    q = normalize_text(query).replace(" ", "")
    k = normalize_text(key).replace(" ", "")
    if not q or not k:
        return 0.0
    if k in q:
        return 0.95
    if q in k:
        return 0.75
    return 0.0


# BM25-like: 87개 symptom key를 짧은 문서로 보고 토큰 점수화
DOC_TOKENS = {k: token_units(k) for k in symptom_vocab}
N_DOCS = len(symptom_vocab)
df = Counter()
for toks in DOC_TOKENS.values():
    for t in set(toks):
        df[t] += 1
idf = {t: math.log((N_DOCS + 1) / (c + 0.5)) for t, c in df.items()}


def bm25_like_score(query: str, key: str) -> float:
    q_toks = token_units(query)
    d_toks = DOC_TOKENS.get(key, [])
    if not q_toks or not d_toks:
        return 0.0
    d_count = Counter(d_toks)
    score = 0.0
    for t in set(q_toks):
        if t in d_count:
            score += idf.get(t, 0.0) * d_count[t]
    # 짧은 key 기준 normalization
    return min(score / 5.0, 1.0)


def vector_like_score(query: str, key: str) -> float:
    # Titan 연결 전 local substitute: char 2-gram cosine
    return cosine_counter(Counter(char_ngrams(query, 2)), Counter(char_ngrams(key, 2)))


def score_query_to_symptom(query: str, symptom_key: str, vector_score: Optional[float] = None) -> Dict[str, float]:
    exact = exact_score(query, symptom_key)
    substring = substring_score(query, symptom_key)
    ngram = jaccard(char_ngrams(query, 2), char_ngrams(symptom_key, 2))
    bm25 = bm25_like_score(query, symptom_key)
    vector = vector_like_score(query, symptom_key) if vector_score is None else vector_score
    final = max(
        0.35 * exact + 0.25 * substring + 0.20 * ngram + 0.10 * bm25 + 0.10 * vector,
        exact,
        substring,
    )
    return {
        "exact": exact,
        "substring": substring,
        "char_ngram": ngram,
        "bm25_like": bm25,
        "vector": vector,
        "score": final,
    }

## 4. Stage 2 local baseline — input만 사용하는 smoke test

`LOCAL_INPUT_ONLY`에서는 테스트셋의 gold `spans`와 `normalized_candidates`를 쓰지 않습니다. 아래 baseline은 단순 rule 기반으로 phrase span과 표준어 변환 후보를 만듭니다.

주의: 이 baseline은 실제 성능용이 아닙니다. 목적은 **gold leakage 없이 전체 파이프라인이 작동하는지 확인**하는 것입니다. 실제 성능은 `AWS_E2E`에서 확인합니다.

In [ ]:

# 강원/구어체 표면형을 표준어에 가깝게 바꾸는 최소 smoke-test 사전.
# 이 값은 테스트용 baseline이며, 운영용 alias 사전으로 확정된 것이 아닙니다.
LOCAL_STANDARD_REPLACEMENTS = {
    "가심": "가슴",
    "맥혀": "막혀",
    "맥힌": "막힌",
    "맥히": "막히",
    "빡혀": "막혀",
    "빡힌": "막힌",
    "목구녕": "목",
    "코빼기": "코",
    "머리깽이": "머리",
    "으찌냑": "어제 저녁",
    "되우": "매우",
    "데우": "매우",
    "하민": "하면서",
    "마카": "모두",
    "마커": "모두",
    "싸래기": "사래",
    "제리제리": "저리저리",
}


def local_standardize_text(text: str) -> str:
    out = normalize_text(text)
    for src, dst in LOCAL_STANDARD_REPLACEMENTS.items():
        out = out.replace(src, dst)
    return out


def suggest_normalized_candidates(source_quote: str, normalized_text: str) -> List[str]:
    text = source_quote + " " + normalized_text
    cands = []
    # 직접 등장하는 symptom key
    for key in symptom_vocab:
        if key.replace(" ", "") in text.replace(" ", ""):
            cands.append(key)

    # 최소 pattern 후보. gold label이 아니라 일반 규칙.
    if re.search(r"기침", text):
        cands.append("기침")
    if re.search(r"가래|객담", text):
        cands.append("가래")
    if re.search(r"피|혈액|각혈", text) and re.search(r"가래|기침|목", text):
        cands.extend(["객혈", "가래"])
    if re.search(r"숨|호흡", text) and re.search(r"차|막|못|힘|답답|빠르|가쁘", text):
        cands.append("호흡곤란")
    if re.search(r"쌕쌕|휘파람", text):
        cands.append("천명음")
    if re.search(r"가슴|흉부", text) and re.search(r"답답", text):
        cands.append("가슴 답답")
    if re.search(r"가슴|흉부", text) and re.search(r"눌|조이|압박", text):
        cands.append("흉부압박감")
    if re.search(r"가슴|흉부", text) and re.search(r"아프|통증|찌르|콕", text):
        cands.append("흉통")
    if re.search(r"열|발열", text):
        cands.append("열")
    if re.search(r"오한|춥|떨", text):
        cands.append("오한")
    if re.search(r"콧물", text):
        cands.append("콧물")
    if re.search(r"재채기", text):
        cands.append("재채기")
    if re.search(r"코", text) and re.search(r"막|맥", text):
        cands.append("코막힘")
    if re.search(r"목", text) and re.search(r"아프|통증|칼칼|따갑", text):
        cands.append("목의 통증")
    if re.search(r"근육", text) and re.search(r"아프|쑤시|통증", text):
        cands.append("근육통")
    if re.search(r"머리", text) and re.search(r"아프|통증", text):
        cands.append("두통")
    if re.search(r"입술|얼굴|손끝", text) and re.search(r"파래|푸르", text):
        cands.append("청색증")
    if re.search(r"두근", text):
        cands.append("가슴 두근거림")

    return [x for x in dict.fromkeys(cands) if x in symptom_set]


def detect_negation(text: str) -> bool:
    return bool(re.search(r"없|아니|안\s|안나|안 나|괜찮", normalize_text(text)))


def rough_split_phrases(text: str) -> List[str]:
    text = normalize_text(text)
    # 문장과 접속 표현 기준으로 과하게 작지 않게 분할
    parts = re.split(r"[.!?。！？]|,|，| 그리고 | 또 | 근데 | 그런데 | 다만 ", text)
    out = []
    for p in parts:
        p = normalize_text(p)
        if not p:
            continue
        # 너무 긴 문장은 '하고/하민/면서' 뒤에서 한 번 더 분리하되, 의미가 깨지지 않도록 제한
        subparts = re.split(r"(?<=하고)\s+|(?<=하민)\s+|(?<=면서)\s+", p)
        for sp in subparts:
            sp = normalize_text(sp)
            if sp:
                out.append(sp)
    return out


def local_input_only_extract_spans(raw_transcript: str) -> Dict[str, Any]:
    spans = []
    for i, phrase in enumerate(rough_split_phrases(raw_transcript), start=1):
        norm = local_standardize_text(phrase)
        span_type = "symptom_candidate"
        if re.search(r"어제|그제|오늘|아침|저녁|밤|부터|며칠|몇일|일주일", norm):
            # onset/course와 증상이 한 phrase에 섞일 수 있으므로 후보도 유지
            span_type = "symptom_candidate" if suggest_normalized_candidates(phrase, norm) else "onset"
        if re.search(r"밤에|아침에|누우면|움직이면|걸으면|심해|덜해", norm):
            span_type = "course" if not suggest_normalized_candidates(phrase, norm) else "symptom_candidate"
        cands = suggest_normalized_candidates(phrase, norm)
        spans.append({
            "span_id": f"local_s{i}",
            "source_quote": phrase,
            "type": span_type,
            "normalization": {
                "normalized_text": norm,
                "normalized_candidates": cands,
                "normalization_reason": "local input-only baseline; gold labels not used",
            },
            "negation_hint": detect_negation(phrase),
            "generated_by": "local_input_only_baseline",
        })
    return {"spans": spans, "generated_by": "local_input_only_baseline"}


def oracle_extract_spans(case: Dict[str, Any]) -> Dict[str, Any]:
    spans = []
    for i, s in enumerate(case.get("spans", []), start=1):
        item = dict(s)
        item.setdefault("span_id", f"gold_s{i}")
        item["generated_by"] = "gold_oracle"
        spans.append(item)
    return {"spans": spans, "generated_by": "gold_oracle"}


def validate_source_quotes(raw: str, spans: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    errors = []
    for s in spans:
        q = s.get("source_quote", "")
        if q and q not in raw:
            errors.append({"span_id": s.get("span_id"), "source_quote": q, "error": "source_quote_not_in_raw"})
    return errors

## 5. AWS 클라이언트 준비

AWS 토글이 꺼져 있으면 호출하지 않습니다.

In [ ]:

import boto3

bedrock_runtime = None
sts_client = None

if USE_AWS_STAGE2 or USE_AWS_EMBEDDINGS or USE_AWS_VERIFIER:
    bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)
    sts_client = boto3.client("sts", region_name=AWS_REGION)
    ident = sts_client.get_caller_identity()
    print("AWS 연결 확인:", {"Account": ident.get("Account"), "Arn": ident.get("Arn")})
else:
    print("AWS 호출 비활성화 상태입니다. 로컬 모드로 진행합니다.")

## 6. Stage 2 AWS Tool Use 함수

`AWS_E2E`에서 `USE_AWS_STAGE2=True`일 때만 호출합니다. 출력은 `source_quote`, `normalized_text`, `normalized_candidates`를 포함해야 합니다.

In [ ]:

SPAN_TOOL = {
    "toolSpec": {
        "name": "extract_q1_spans_and_normalization",
        "description": "Extract Q1 spans and normalization candidates.",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "spans": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "span_id": {"type": "string"},
                                "source_quote": {"type": "string"},
                                "type": {"type": "string"},
                                "normalized_text": {"type": "string"},
                                "normalized_candidates": {
                                    "type": "array",
                                    "items": {"type": "string"}
                                },
                                "negation_hint": {"type": "boolean"}
                            },
                            "required": [
                                "span_id",
                                "source_quote",
                                "type",
                                "normalized_text",
                                "normalized_candidates"
                            ]
                        }
                    }
                },
                "required": ["spans"]
            }
        }
    }
}


def _extract_tool_input_from_converse(response: Dict[str, Any], tool_name: str) -> Dict[str, Any]:
    for block in response.get("output", {}).get("message", {}).get("content", []):
        if "toolUse" in block and block["toolUse"].get("name") == tool_name:
            return block["toolUse"].get("input", {})
    raise ValueError(f"toolUse {tool_name} not found: {response}")


def bedrock_extract_spans(raw_transcript: str, allowed_symptoms: List[str]) -> Dict[str, Any]:
    assert bedrock_runtime is not None
    prompt = f"""
당신은 문진톡톡 Q1 전처리기입니다.
환자 발화에서 phrase span을 추출하고, 사투리/구어체 표현을 표준어로 변환하며, symptom vocabulary 검색용 후보를 생성하세요.

환자 원문:
{raw_transcript}

허용된 표준 증상 key 목록:
{json.dumps(allowed_symptoms, ensure_ascii=False)}

규칙:
1. source_quote는 반드시 환자 원문에 실제 존재하는 부분 문자열이어야 합니다.
2. source_quote를 삭제하거나 표준어로 덮어쓰지 마세요.
3. normalized_text는 source_quote의 표준어 변환문입니다.
4. normalized_candidates는 허용된 표준 증상 key와 가까운 검색 후보입니다.
5. 질환명, 진단명, 검사 권고, 처방, 복약 판단을 생성하지 마세요.
6. 애매한 표현은 후보로만 두고 확정하지 마세요.
7. 명시적 부정 표현은 negation_hint=true로 표시하세요.

extract_q1_spans_and_normalization 도구를 호출하세요.
""".strip()

    response = bedrock_runtime.converse(
        modelId=LLM_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        toolConfig={
            "tools": [SPAN_TOOL],
            "toolChoice": {"tool": {"name": "extract_q1_spans_and_normalization"}},
        },
        inferenceConfig={"temperature": 0.0, "maxTokens": 1600},
    )
    payload = _extract_tool_input_from_converse(response, "extract_q1_spans_and_normalization")
    payload["generated_by"] = "aws_bedrock_stage2_tool_use"
    for s in payload.get("spans", []):
        s["generated_by"] = "aws_bedrock_stage2_tool_use"
    time.sleep(AWS_SLEEP_SEC)
    return payload

## 7. Stage 3 — Ensemble Retrieval + optional Titan embedding

`source_quote`, `normalized_text`, `normalized_candidates`를 모두 query로 사용합니다.

중요: 여기서 꺼내는 `candidate_symptom_matches`는 **질환 후보가 아니라 표준 증상 후보 shortlist**입니다.

In [ ]:

EMBED_CACHE_PATH = CACHE_DIR / f"embedding_cache_{EMBEDDING_MODEL_ID.replace(':','_')}_{EMBEDDING_DIMENSIONS}.json"
if EMBED_CACHE_PATH.exists():
    embedding_cache = json.loads(EMBED_CACHE_PATH.read_text(encoding="utf-8"))
else:
    embedding_cache = {}


def save_embedding_cache():
    EMBED_CACHE_PATH.write_text(json.dumps(embedding_cache, ensure_ascii=False), encoding="utf-8")


def titan_embed(text: str) -> List[float]:
    assert bedrock_runtime is not None
    text = normalize_text(text)
    if not text:
        return []
    cache_key = f"{EMBEDDING_MODEL_ID}|{EMBEDDING_DIMENSIONS}|{text}"
    if cache_key in embedding_cache:
        return embedding_cache[cache_key]
    body = {"inputText": text, "dimensions": EMBEDDING_DIMENSIONS, "normalize": True}
    resp = bedrock_runtime.invoke_model(
        modelId=EMBEDDING_MODEL_ID,
        contentType="application/json",
        accept="application/json",
        body=json.dumps(body, ensure_ascii=False).encode("utf-8"),
    )
    payload = json.loads(resp["body"].read().decode("utf-8"))
    vec = payload.get("embedding")
    if vec is None:
        raise ValueError(f"Titan embedding not found in payload: {payload}")
    embedding_cache[cache_key] = vec
    if len(embedding_cache) % 25 == 0:
        save_embedding_cache()
    time.sleep(AWS_SLEEP_SEC)
    return vec


def cosine_np(a: List[float], b: List[float]) -> float:
    if not a or not b:
        return 0.0
    aa, bb = np.array(a, dtype=float), np.array(b, dtype=float)
    denom = np.linalg.norm(aa) * np.linalg.norm(bb)
    if denom == 0:
        return 0.0
    return float(np.dot(aa, bb) / denom)


def build_queries_for_span(span: Dict[str, Any]) -> List[str]:
    queries = []
    sq = span.get("source_quote", "")
    if sq:
        queries.append(sq)
    norm = (span.get("normalization") or {}).get("normalized_text", "")
    if norm:
        queries.append(norm)
    for c in (span.get("normalization") or {}).get("normalized_candidates", []) or []:
        if c:
            queries.append(c)
    # 중복 제거
    return list(dict.fromkeys(normalize_text(q) for q in queries if normalize_text(q)))


symptom_embeddings = {}
if USE_AWS_EMBEDDINGS:
    print("Pre-embedding symptom vocabulary with Titan...")
    for key in tqdm(symptom_vocab):
        symptom_embeddings[key] = titan_embed(key)
    save_embedding_cache()
    print("symptom embeddings ready:", len(symptom_embeddings))
else:
    print("Titan 비활성화: local vector-like score 사용")


def score_query_to_symptom_with_optional_titan(query: str, symptom_key: str) -> Dict[str, float]:
    if USE_AWS_EMBEDDINGS:
        q_vec = titan_embed(query)
        s_vec = symptom_embeddings.get(symptom_key) or titan_embed(symptom_key)
        vector = cosine_np(q_vec, s_vec)
        base = score_query_to_symptom(query, symptom_key, vector_score=vector)
    else:
        base = score_query_to_symptom(query, symptom_key)
    return base


def retrieve_for_span(span: Dict[str, Any], top_k: int = TOP_K) -> Dict[str, Any]:
    queries = build_queries_for_span(span)
    by_symptom = {}
    for symptom_key in symptom_vocab:
        best = None
        for q in queries:
            sc = score_query_to_symptom_with_optional_titan(q, symptom_key)
            row = {
                "symptom_name": symptom_key,
                "score": sc["score"],
                "best_query": q,
                "method_scores": sc,
            }
            if best is None or row["score"] > best["score"]:
                best = row
        if best:
            by_symptom[symptom_key] = best
    shortlist = sorted(by_symptom.values(), key=lambda x: x["score"], reverse=True)[:top_k]
    return {
        "span_id": span.get("span_id"),
        "source_quote": span.get("source_quote"),
        "queries_used": queries,
        "candidate_symptom_matches": shortlist,
    }


def run_retrieval(spans: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    results = []
    for span in spans:
        if span.get("type") not in {"symptom_candidate", "symptom", "other"}:
            continue
        results.append(retrieve_for_span(span, TOP_K))
    if USE_AWS_EMBEDDINGS:
        save_embedding_cache()
    return results

## 8. Stage 4 Verifier

Local verifier는 baseline입니다. 실제 검증은 Bedrock verifier를 켜고 확인합니다.

In [ ]:

VERIFIER_TOOL = {
    "toolSpec": {
        "name": "verify_q1_symptom_matching",
        "description": "Q1 span, 표준어 변환, retrieval shortlist를 검증해 matched/denied/needs_review/unmatched를 반환한다. 질환명/진단/처방 생성 금지.",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "matched_symptoms": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "span_id": {"type": "string"},
                                "source_quote": {"type": "string"},
                                "normalized_text": {"type": "string"},
                                "symptom_name": {"type": "string"},
                                "status": {"type": "string", "description": "matched, denied, needs_review, unmatched 중 하나"},
                                "confidence": {"type": "string", "description": "high, medium, low 중 하나"},
                                "confidence_score": {"type": "number"},
                                "verification_reason": {"type": "string"}
                            },
                            "required": ["span_id", "source_quote", "symptom_name", "status", "confidence"]
                        }
                    },
                    "unmatched_spans": {"type": "array", "items": {"type": "object"}},
                    "needs_review": {"type": "array", "items": {"type": "object"}}
                },
                "required": ["matched_symptoms", "unmatched_spans", "needs_review"]
            }
        }
    }
}


def local_verify_heuristic(spans: List[Dict[str, Any]], retrieval_results: List[Dict[str, Any]]) -> Dict[str, Any]:
    by_span = {r["span_id"]: r for r in retrieval_results}
    matched = []
    unmatched = []
    needs_review = []
    for span in spans:
        if span.get("type") not in {"symptom_candidate", "symptom", "other"}:
            continue
        r = by_span.get(span.get("span_id"))
        shortlist = (r or {}).get("candidate_symptom_matches", [])
        if not shortlist:
            unmatched.append({"span_id": span.get("span_id"), "source_quote": span.get("source_quote"), "reason": "no_retrieval_candidate"})
            continue
        top = shortlist[0]
        score = float(top.get("score", 0.0))
        status = "matched"
        confidence = "high"
        if span.get("negation_hint"):
            status = "denied"
            confidence = "high" if score >= 0.55 else "medium"
        elif score >= 0.72:
            status = "matched"
            confidence = "high"
        elif score >= 0.48:
            status = "needs_review"
            confidence = "medium"
            needs_review.append({
                "span_id": span.get("span_id"),
                "source_quote": span.get("source_quote"),
                "candidate_symptom_name": top.get("symptom_name"),
                "reason": "local verifier: score in review range",
            })
        else:
            status = "unmatched"
            confidence = "low"
            unmatched.append({
                "span_id": span.get("span_id"),
                "source_quote": span.get("source_quote"),
                "reason": "local verifier: score below threshold",
            })
        matched.append({
            "span_id": span.get("span_id"),
            "source_quote": span.get("source_quote"),
            "normalized_text": (span.get("normalization") or {}).get("normalized_text", ""),
            "symptom_name": top.get("symptom_name"),
            "status": status,
            "confidence": confidence,
            "confidence_score": round(score, 4),
            "verification_reason": "local heuristic verifier; not clinical validation",
            "shortlist": shortlist,
        })
    return {"matched_symptoms": matched, "unmatched_spans": unmatched, "needs_review": needs_review}


def bedrock_verify(raw_transcript: str, spans: List[Dict[str, Any]], retrieval_results: List[Dict[str, Any]], allowed_symptoms: List[str]) -> Dict[str, Any]:
    assert bedrock_runtime is not None
    prompt_payload = {
        "raw_transcript": raw_transcript,
        "allowed_symptom_keys": allowed_symptoms,
        "spans": spans,
        "retrieval_results": retrieval_results,
    }
    prompt = f"""
당신은 문진톡톡 Q1 증상 매칭 검증기입니다.
아래 파싱/정규화/retrieval 결과를 검증하세요.

입력 JSON:
{json.dumps(prompt_payload, ensure_ascii=False)}

규칙:
1. allowed_symptom_keys 밖의 symptom_name을 생성하지 마세요.
2. 질환명, 진단명, 검사 권고, 처방, 복약 판단을 생성하지 마세요.
3. source_quote는 원문에 실제 있는 표현이어야 합니다.
4. candidate_symptom_matches는 참고 shortlist일 뿐이며, 애매하면 needs_review로 두세요.
5. 명시적 부정 표현이면 denied로 처리하세요.
6. 최종 status는 matched, denied, needs_review, unmatched 중 하나입니다.

verify_q1_symptom_matching 도구를 호출하세요.
""".strip()
    response = bedrock_runtime.converse(
        modelId=LLM_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        toolConfig={
            "tools": [VERIFIER_TOOL],
            "toolChoice": {"tool": {"name": "verify_q1_symptom_matching"}},
        },
        inferenceConfig={"temperature": 0.0, "maxTokens": 1800},
    )
    payload = _extract_tool_input_from_converse(response, "verify_q1_symptom_matching")
    time.sleep(AWS_SLEEP_SEC)
    return payload

## 9. Stage 5 Safety flag 병렬 감지

테스트용 rule입니다. 최종 임상 기준이 아닙니다. 질병이 아니라 **증상·표현** 기준으로만 표시합니다.

In [ ]:

TEST_SAFETY_RULES = [
    {
        "flag_id": "safety:hemoptysis_expression",
        "symptom_keys": ["객혈"],
        "keywords": ["가래에 피", "피가래", "피 섞", "피가 섞", "각혈", "기침할 때 피", "피가 조금", "피가 보여"],
        "message": "객혈 의심 표현이 있어 의료진 우선 확인 필요",
    },
    {
        "flag_id": "safety:severe_dyspnea_expression",
        "symptom_keys": ["호흡곤란", "운동 시 호흡곤란"],
        "keywords": ["숨을 못 쉬", "말하기도 힘", "숨이 너무 차", "숨이 빡힌", "질식할 것", "숨이 막힐 때"],
        "message": "심한 호흡곤란 표현이 있어 의료진 우선 확인 필요",
    },
    {
        "flag_id": "safety:cyanosis_expression",
        "symptom_keys": ["청색증"],
        "keywords": ["입술이 파래", "파래지는", "얼굴이 파래", "손끝이 파래"],
        "message": "청색증 관련 표현이 있어 의료진 우선 확인 필요",
    },
    {
        "flag_id": "safety:chest_pain_expression",
        "symptom_keys": ["흉통", "흉부압박감"],
        "keywords": ["가슴이 콕", "콕 찌르", "가슴이 갑자기", "흉통", "가슴이 눌", "흉부압박감", "담 결린"],
        "message": "흉통 또는 흉부압박감 표현이 있어 의료진 확인 필요",
    },
    {
        "flag_id": "safety:choking_expression",
        "symptom_keys": ["사래걸림", "질식"],
        "keywords": ["사래", "싸래기", "질식", "켁켁"],
        "message": "사래/질식 관련 표현이 있어 의료진 확인 필요",
    },
    {
        "flag_id": "safety:altered_mental_status",
        "symptom_keys": ["의식 변화", "의식 저하"],
        "keywords": ["의식이 흐", "정신이 흐", "깨우기", "혼미"],
        "message": "의식 변화 표현이 있어 의료진 우선 확인 필요",
    },
]


def detect_safety_flags(raw_transcript: str, spans: List[Dict[str, Any]], verifier_result: Dict[str, Any]) -> List[Dict[str, Any]]:
    texts = [normalize_text(raw_transcript)]
    for s in spans:
        texts.append(normalize_text(s.get("source_quote", "")))
        texts.append(normalize_text((s.get("normalization") or {}).get("normalized_text", "")))
        for c in (s.get("normalization") or {}).get("normalized_candidates", []) or []:
            texts.append(normalize_text(c))
    matched_names = [m.get("symptom_name", "") for m in verifier_result.get("matched_symptoms", []) if m.get("status") in {"matched", "needs_review"}]
    texts.extend(matched_names)
    joined = "\n".join(t for t in texts if t)

    flags = []
    for rule in TEST_SAFETY_RULES:
        keyword_hit = next((kw for kw in rule["keywords"] if kw in joined), None)
        symptom_hit = next((sk for sk in rule["symptom_keys"] if sk in matched_names), None)
        if keyword_hit or symptom_hit:
            flags.append({
                "flag_id": rule["flag_id"],
                "trigger_keyword": keyword_hit,
                "trigger_symptom": symptom_hit,
                "display_level": "staff_check_required",
                "message": rule["message"],
            })
    # flag_id 중복 제거
    dedup = {}
    for f in flags:
        dedup[f["flag_id"]] = f
    return list(dedup.values())

## 10. Stage 6 원페이퍼 입력 JSON 생성

In [ ]:

def build_q1_onepager_input(case: Dict[str, Any], spans: List[Dict[str, Any]], verifier_result: Dict[str, Any], safety_flags: List[Dict[str, Any]]) -> Dict[str, Any]:
    mapping = []
    for m in verifier_result.get("matched_symptoms", []):
        mapping.append({
            "patient_expression": m.get("source_quote", ""),
            "standardized_expression": m.get("normalized_text", ""),
            "standard_symptom": m.get("symptom_name", ""),
            "status": m.get("status", ""),
            "confidence": m.get("confidence", ""),
            "confidence_score": m.get("confidence_score"),
        })
    return {
        "case_id": case.get("case_id"),
        "raw_transcript": case.get("input", ""),
        "speech_quote": case.get("input", ""),
        "symptom_mapping": mapping,
        "needs_review": verifier_result.get("needs_review", []),
        "unmatched_spans": verifier_result.get("unmatched_spans", []),
        "safety_flags": safety_flags,
    }

## 11. 전체 파이프라인 실행

중요: `PIPELINE_MODE != "ORACLE_RETRIEVAL_COMPONENT"`이면 inference에 gold span/normalization을 넣지 않습니다.

In [ ]:

def stage2_extract(case: Dict[str, Any]) -> Dict[str, Any]:
    raw_transcript = case["input"]
    if PIPELINE_MODE == "ORACLE_RETRIEVAL_COMPONENT":
        return oracle_extract_spans(case)
    if USE_AWS_STAGE2:
        return bedrock_extract_spans(raw_transcript, symptom_vocab)
    return local_input_only_extract_spans(raw_transcript)


def process_case(case: Dict[str, Any]) -> Dict[str, Any]:
    raw_transcript = case["input"]

    # Stage 1
    stage1 = {"raw_transcript": raw_transcript, "confirmed_text": raw_transcript}

    # Stage 2/2.5
    stage2 = stage2_extract(case)
    spans = stage2.get("spans", [])
    quote_errors = validate_source_quotes(raw_transcript, spans)

    # Gold leakage guard
    if PIPELINE_MODE != "ORACLE_RETRIEVAL_COMPONENT":
        leaked = [s for s in spans if s.get("generated_by") == "gold_oracle"]
        if leaked:
            raise RuntimeError("Gold span leakage detected in non-oracle mode")

    # Stage 3
    retrieval_results = run_retrieval(spans)

    # Stage 4
    if USE_AWS_VERIFIER:
        verifier_result = bedrock_verify(raw_transcript, spans, retrieval_results, symptom_vocab)
    else:
        verifier_result = local_verify_heuristic(spans, retrieval_results)

    # Stage 5
    safety_flags = detect_safety_flags(raw_transcript, spans, verifier_result)

    # Stage 6
    onepager_input = build_q1_onepager_input(case, spans, verifier_result, safety_flags)

    # expected labels are included only after inference, for evaluation.
    return {
        "case_id": case.get("case_id"),
        "input": raw_transcript,
        "expected_symptoms": case.get("expected_symptoms", []),
        "expected_denied": case.get("expected_denied", []),
        "expected_needs_review": case.get("expected_needs_review", []),
        "expected_safety_flags": case.get("expected_safety_flags", []),
        "pipeline_mode": PIPELINE_MODE,
        "stage1": stage1,
        "stage2_generated_by": stage2.get("generated_by"),
        "stage2_spans": spans,
        "source_quote_errors": quote_errors,
        "stage3_retrieval": retrieval_results,
        "stage4_verifier": verifier_result,
        "stage5_safety_flags": safety_flags,
        "stage6_onepager_input": onepager_input,
    }

cases_to_run = test_cases if MAX_CASES is None else test_cases[:MAX_CASES]
print("실행 케이스 수:", len(cases_to_run))
print("PIPELINE_MODE:", PIPELINE_MODE)

results = []
for case in tqdm(cases_to_run):
    try:
        results.append(process_case(case))
    except Exception as e:
        results.append({"case_id": case.get("case_id"), "input": case.get("input"), "error": repr(e), "pipeline_mode": PIPELINE_MODE})

out_path = OUTPUT_DIR / f"q1_pipeline_results_{PIPELINE_MODE.lower()}.json"
out_path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장:", out_path)
print("에러 수:", sum(1 for r in results if "error" in r))

# 첫 결과 확인
print(json.dumps(results[0], ensure_ascii=False, indent=2)[:6000])

## 12. 평가 지표

평가 단계에서만 gold expected label을 사용합니다.

In [ ]:

def set_recall(expected: List[str], predicted: List[str]) -> Optional[float]:
    expected = set(expected or [])
    predicted = set(predicted or [])
    if not expected:
        return None
    return len(expected & predicted) / len(expected)


def evaluate_results(results: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for r in results:
        if "error" in r:
            rows.append({
                "case_id": r.get("case_id"),
                "input": r.get("input"),
                "error": r.get("error"),
                "symptom_recall": 0.0,
                "denied_recall": 0.0,
                "safety_recall": 0.0,
                "source_quote_error_count": None,
            })
            continue
        matched = r["stage4_verifier"].get("matched_symptoms", [])
        pred_matched = [m["symptom_name"] for m in matched if m.get("status") == "matched"]
        pred_denied = [m["symptom_name"] for m in matched if m.get("status") == "denied"]
        pred_review = [m["symptom_name"] for m in matched if m.get("status") == "needs_review"]
        pred_safety = [f["flag_id"] for f in r.get("stage5_safety_flags", [])]

        exp_sym = r.get("expected_symptoms", [])
        exp_denied = r.get("expected_denied", [])
        exp_safety = r.get("expected_safety_flags", [])

        rows.append({
            "case_id": r.get("case_id"),
            "input": r.get("input"),
            "stage2_generated_by": r.get("stage2_generated_by"),
            "expected_symptoms": ", ".join(exp_sym),
            "pred_matched": ", ".join(pred_matched),
            "pred_needs_review": ", ".join(pred_review),
            "symptom_recall": set_recall(exp_sym, pred_matched),
            "expected_denied": ", ".join(exp_denied),
            "pred_denied": ", ".join(pred_denied),
            "denied_recall": set_recall(exp_denied, pred_denied),
            "expected_safety_flags": ", ".join(exp_safety),
            "pred_safety_flags": ", ".join(pred_safety),
            "safety_recall": set_recall(exp_safety, pred_safety),
            "source_quote_error_count": len(r.get("source_quote_errors", [])),
        })
    return pd.DataFrame(rows)


eval_df = evaluate_results(results)
eval_path = OUTPUT_DIR / f"q1_pipeline_eval_{PIPELINE_MODE.lower()}.csv"
eval_df.to_csv(eval_path, index=False, encoding="utf-8-sig")
print("저장:", eval_path)

summary = {
    "pipeline_mode": PIPELINE_MODE,
    "n_cases": len(eval_df),
    "mean_symptom_recall_ignore_na": float(eval_df["symptom_recall"].dropna().mean()) if eval_df["symptom_recall"].notna().any() else None,
    "mean_denied_recall_ignore_na": float(eval_df["denied_recall"].dropna().mean()) if eval_df["denied_recall"].notna().any() else None,
    "mean_safety_recall_ignore_na": float(eval_df["safety_recall"].dropna().mean()) if eval_df["safety_recall"].notna().any() else None,
    "source_quote_errors_total": int(eval_df["source_quote_error_count"].fillna(0).sum()),
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
display(eval_df.head(20))

## 13. 실패 케이스 확인

`LOCAL_INPUT_ONLY`의 실패는 자연스럽습니다. 이 모드는 AWS Stage 2/2.5 없이 rule baseline으로만 돌리기 때문입니다. 여기서 볼 것은 다음입니다.

- 어떤 표현이 표준어 변환 없이는 실패하는가
- 어떤 표현이 vector가 필요해 보이는가
- safety rule이 누락/과잉되는가
- Bedrock Stage 2를 켰을 때 개선되는가

In [ ]:

fail_df = eval_df.copy()
for col in ["symptom_recall", "denied_recall", "safety_recall"]:
    fail_df[col] = fail_df[col].fillna(1.0)
fail_df = fail_df[
    (fail_df["symptom_recall"] < 1.0)
    | (fail_df["denied_recall"] < 1.0)
    | (fail_df["safety_recall"] < 1.0)
    | (fail_df["source_quote_error_count"].fillna(0) > 0)
]
print("실패/검토 케이스 수:", len(fail_df))
display(fail_df)

# 특정 case_id 상세 확인용
CASE_ID_TO_INSPECT = results[0]["case_id"] if results else None
if CASE_ID_TO_INSPECT:
    detail = next((r for r in results if r.get("case_id") == CASE_ID_TO_INSPECT), None)
    print("INSPECT:", CASE_ID_TO_INSPECT)
    print(json.dumps(detail, ensure_ascii=False, indent=2)[:10000])

## 14. 권장 실험 순서

### A. 로컬 smoke test

```python
PIPELINE_MODE = "LOCAL_INPUT_ONLY"
USE_AWS_STAGE2 = False
USE_AWS_EMBEDDINGS = False
USE_AWS_VERIFIER = False
MAX_CASES = 10
```

목적: gold leakage 없이 코드 흐름이 도는지 확인.

### B. Oracle retrieval component test

```python
PIPELINE_MODE = "ORACLE_RETRIEVAL_COMPONENT"
```

목적: gold span/정규화가 주어졌을 때 Stage 3 retrieval이 동작하는지 확인. 이 결과를 end-to-end 성능으로 말하지 않음.

### C. AWS E2E test

```python
PIPELINE_MODE = "AWS_E2E"
USE_AWS_STAGE2 = True
USE_AWS_EMBEDDINGS = False
USE_AWS_VERIFIER = False
MAX_CASES = 3
```

이후 Titan/Verifier를 하나씩 켭니다.

## 15. AWS 자격 및 모델 접근 확인 셀

아래 셀은 필요할 때만 `True`로 바꾸세요.

In [ ]:

CHECK_AWS_IDENTITY = False
LIST_BEDROCK_MODELS = False

if CHECK_AWS_IDENTITY:
    import boto3
    sts = boto3.client("sts", region_name=AWS_REGION)
    print(sts.get_caller_identity())
else:
    print("CHECK_AWS_IDENTITY=False")

if LIST_BEDROCK_MODELS:
    import boto3
    bedrock = boto3.client("bedrock", region_name=AWS_REGION)
    models = bedrock.list_foundation_models().get("modelSummaries", [])
    df_models = pd.DataFrame([{
        "modelId": m.get("modelId"),
        "providerName": m.get("providerName"),
        "modelName": m.get("modelName"),
        "inputModalities": m.get("inputModalities"),
        "outputModalities": m.get("outputModalities"),
    } for m in models])
    display(df_models.sort_values(["providerName", "modelId"]))
else:
    print("LIST_BEDROCK_MODELS=False")